# Lead Generation — Monthly Upload FileRestructured from `messt data processing.py`. See `CODE_REVIEW.md` in this folder for thefull list of issues this version fixes (referenced below as **C1–C7** and **M1–M17**).**What it produces**| Deliverable | Rows | Content ||---|---|---|| `leadgen_uploadfile_<date>_p1.csv` / `_p2.csv` | All customers (ineligible have PII masked) | 32 core columns + 17 **deposit** columns from `depo_info` || `leadgen_uploadfile_<date>.parquet` | Eligible only, virtual branches excluded | 32 core columns + 7 **financing** columns. **Carries unmasked PII.** |**How to run**1. Edit **§2 CONFIG** — that is the only cell you change each month.2. Kernel → Restart & Run All. It is safe to do that; no cell depends on being run twice.3. Section §3 fails fast if any configured file is missing, before the long reads start.**Rules for anyone editing this notebook**- One cell, one step. Never mutate a variable that a later cell also mutates.- No path outside §2. If you find one, it is a bug.- Never rename columns positionally (`df.columns = [...]`). Always `rename(columns={...})` — see **C1**.- Never `except Exception`. Catch the specific error or let it fail.- Every join on `CIFNO` goes through `merge_1to1`.

## 1. Imports and display options

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime as dt

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 1000)
pd.options.display.float_format = '{:.2f}'.format

print(f"pandas {pd.__version__}")

## 2. CONFIG — the only cell you edit each monthEvery path is a full path, so nothing is tied to one drive letter or folder layout.Each entry carries a remark naming the **file used in the original version**, so you cansee the vintage you are replacing and the naming pattern to follow.Refresh cadence:- **Core data** — new extract in the first week of every month.- **Contact data** — deliberately lagged; see the note in that block.- **Supplementary** — only when the owner named in the remark sends a new file.- **Lookups** — rarely change.

In [ ]:
CONFIG = {
    # ==================================================================
    # RUN SETTINGS
    # ==================================================================
    # ORIGINAL: CSV filenames hardcoded "20260910" while the parquet used
    #           dt.today() -- the two deliverables could carry different
    #           dates, and re-running overwrote the previous month.  [C6]
    "run_date":   dt.today().strftime('%Y%m%d'),

    "output_dir": r"D:\Documents\Data\CustellAI\File_upload",

    # ORIGINAL: always exactly 2 parts, split with len() of the wrong frame. [C2]
    "csv_parts":  2,

    # Branches excluded from the PARQUET only (not the CSV).
    # ORIGINAL: hardcoded inline near the end of the notebook. [M5]
    "parquet_excluded_branches": [
        'Mobile Banking App Virtual Branch',
        'Internet Banking Virtual Branch',
        'IFSC, JOHOR BAHRU',
    ],

    # Leave False to keep the parquet's financing columns in their original
    # Title Case names ('AR-RAHNU Balance', 'CC Limit', ...). Set True only
    # after confirming no downstream consumer depends on those names. [M12]
    "normalise_parquet_names": False,

    # ==================================================================
    # CORE DATA  --  refresh in the first week of every month
    # ==================================================================
    # ORIGINAL: casatia_2608_AUG26.csv        pattern: casatia_YYMM_MMMYY.csv
    "casa":        r"D:\Documents\Data\CASATIA\2026\casatia_2608_AUG26.csv",

    # ORIGINAL: NON-CASATIA_2608_AUG26.csv    pattern: NON-CASATIA_YYMM_MMMYY.csv
    "nonc":        r"D:\Documents\Data\Non-CASATIA\2026\NON-CASATIA_2608_AUG26.csv",

    # ORIGINAL: 2026-08 Fin.csv               pattern: YYYY-MM Fin.csv
    "fina":        r"D:\Documents\Data\Financing\2026\2026-08 Fin.csv",

    # ORIGINAL: 202608_9000_MasterFile Credit Card.csv
    #                                         pattern: YYYYMM_9000_MasterFile Credit Card.csv
    "cc_master":   r"D:\Documents\Data\Credit Card\2026\202608_9000_MasterFile Credit Card.csv",

    # ==================================================================
    # CONTACT DATA  --  intentionally lagged, NOT the current month
    # ==================================================================
    # The current-month extract is not always usable for phone and email, so
    # these are a separate monthly judgement call rather than a copy of the
    # core paths.
    # ORIGINAL: buried as hardcoded arguments in the email_phone(...) call,
    #           pointing at APR26 while the core data was AUG26.
    "contact_casa": r"D:\Documents\Data\CASATIA\2026\casatia_2604_APR26.csv",
    "contact_fina": r"D:\Documents\Data\Financing\2026\2026-04 Fin.csv",

    # ==================================================================
    # SUPPLEMENTARY  --  refresh when the owner sends a new file
    # ==================================================================
    # ORIGINAL: MAR 26 ACCOUNT HOLDER_ACE_BIMB_ACCOUNTS_EXPORT_2026-04-05.xlsx
    "gold":        r"D:\Documents\Data\Gold\2603\MAR 26 ACCOUNT HOLDER_ACE_BIMB_ACCOUNTS_EXPORT_2026-04-05.xlsx",   # owner: Farisya

    # ORIGINAL: fin_customer_empl_march26.csv
    "empl":        r"D:\Documents\Data\Miscellaneous\2604\Farisya March 2026\fin_customer_empl_march26.csv",        # owner: Farisya

    # ORIGINAL: 202603 Unique IBMB Customer Listing.csv
    "ibmb":        r"D:\Documents\Data\IBMB\2603\202603 Unique IBMB Customer Listing.csv",                          # owner: Farisya

    # ORIGINAL: SCReport-31-12-2025.xlsx
    # Column meanings confirmed correct: 'Total Holdings' -> unit_trust_balance,
    # 'AUM' -> unit_trust_unit. Renamed by name now, so the order in the sheet
    # no longer matters. [C1]
    "ut_aum":      r"D:\Documents\Data\Miscellaneous\2603\UT Balance\SCReport-31-12-2025.xlsx",                     # owner: Farith

    # ORIGINAL: 6month_salary_cr.csv
    "salary_cr":   r"D:\Downloads\2601\CustellAI\6month_salary_cr.csv",                                            # owner: Solah

    # ORIGINAL: query_download_data.csv
    "query_dwnld": r"D:\Downloads\2604\Custell_Meta_Data\query_download_data.csv",                                 # owner: Solah

    # ORIGINAL: Debit_Cards_202512.csv        pattern: Debit_Cards_YYYYMM.csv
    "debit":       r"D:\Documents\Data\Debit_Cards\Debit_Cards_202512.csv",                                        # owner: Aini

    # ORIGINAL: Product_holding_022026_unique_new.csv
    "ph":          r"D:\Documents\Data\Product Holding\Product_holding_022026_unique_new.csv",                      # owner: Ira

    # ORIGINAL: cc_customer.csv
    "cc":          r"D:\Downloads\2601\CustellAI\cc_customer.csv",                                                  # owner: TBD

    # ORIGINAL: hardcoded inline mid-notebook, NOT in DEFAULT_PATHS. Supplies
    #           all 17 deposit columns of the CSV deliverable. [M7]
    "depo_info":   r"D:\Downloads\2606\depo_info.csv",                                                              # owner: TBD

    # ==================================================================
    # LOOKUPS  --  change rarely
    # ==================================================================
    # ORIGINAL: Branch Lookup_Feb2026.csv
    "branch_lookup": r"D:\Documents\Data\Lookups\Branch Lookup_Feb2026.csv",

    # ORIGINAL: occupation_simplified.csv
    "occ_lookup":    r"D:\Documents\Data\Lookups\occupation_simplified.csv",
}

# Credit-card columns expected in the cc file, best first.
# ORIGINAL: melt() treated EVERY non-id column as a card type, so one extra
#           column upstream silently corrupted cc_type. Now explicit. [M8]
CC_TYPE_ORDER = ['Mastercard World', 'Visa Infinite', 'Gold CC', 'Standard CC']

PATH_KEYS = ['casa', 'nonc', 'fina', 'cc_master', 'contact_casa', 'contact_fina',
             'gold', 'empl', 'ibmb', 'ut_aum', 'salary_cr', 'query_dwnld',
             'debit', 'ph', 'cc', 'depo_info', 'branch_lookup', 'occ_lookup']

## 3. Path check — fail before the long readsA full run takes minutes. This costs a second and tells you immediately if a path in §2is stale, instead of failing halfway through.

In [ ]:
missing = [k for k in PATH_KEYS if not os.path.isfile(CONFIG[k])]
for k in missing:
    print(f"  MISSING  {k:<14s} ->  {CONFIG[k]}")

if missing:
    raise FileNotFoundError(f"{len(missing)} configured input file(s) not found. Fix CONFIG (§2) first.")
if not os.path.isdir(CONFIG['output_dir']):
    raise NotADirectoryError(f"output_dir does not exist: {CONFIG['output_dir']}")

print(f"All {len(PATH_KEYS)} input files found. Output directory OK.")

## 4. HelpersFour helpers, each with one job. Two of them exist specifically to stop bugs the originalversion had:- **`read_table`** renames **by name**, and raises if an expected column is absent.  The original assigned names positionally after `usecols` — and pandas returns columns in  *file* order, not the order you listed. One reordered upstream file put emails in a phone  column with no error. **[C1]**- **`merge_1to1`** de-duplicates the right frame and asserts the row count did not change,  so a join can never silently fan out. **[C5]**`clean_cif` and `format_phone` are each defined **once**. The original had two `format_phone`implementations that disagreed, so the CSV and parquet formatted phones differently. **[C3]**

In [ ]:
# pyarrow is faster but optional -- the original assumed it was installed.
try:
    from pyarrow.lib import ArrowInvalid
    _HAS_PYARROW = True
    _DECODE_ERRORS = (UnicodeDecodeError, ArrowInvalid)
except ImportError:
    _HAS_PYARROW = False
    _DECODE_ERRORS = (UnicodeDecodeError,)
    print("note: pyarrow not installed -- falling back to the default CSV engine")


def _read_csv(path, columns=None):
    """CSV read with encoding fallback. Catches ONLY decode errors -- a missing
    column or a bad path still raises its real error.
    ORIGINAL: `except (UnicodeDecodeError, Exception)` swallowed everything and
    retried a column error as an encoding problem. [M2]"""
    if _HAS_PYARROW:
        try:
            return pd.read_csv(path, usecols=columns, engine='pyarrow')
        except _DECODE_ERRORS:
            pass
    for enc in (None, 'ISO-8859-1', 'cp1252'):
        try:
            return pd.read_csv(path, usecols=columns, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError('utf-8', b'', 0, 1,
                             f"could not decode {os.path.basename(path)} with utf-8/latin1/cp1252")


def read_table(path, columns=None, rename=None):
    """Read csv / xlsx / parquet, then rename BY NAME.

    columns : list of source column names to read (order is irrelevant)
    rename  : {source_name: target_name}; raises KeyError if a key is missing
    """
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.xlsx', '.xls', '.xlsm'):
        df = pd.read_excel(path, usecols=columns)
    elif ext == '.parquet':
        df = pd.read_parquet(path, columns=columns)
    else:
        df = _read_csv(path, columns)

    if rename:
        absent = sorted(set(rename) - set(df.columns))
        if absent:
            raise KeyError(f"{os.path.basename(path)}: expected column(s) not found: {absent}. "
                           f"File has: {sorted(df.columns)}")
        df = df.rename(columns=rename)
    return df


def clean_cif(series):
    """CIFNO as nullable Int64 everywhere -- one dtype for the join key. [M14]"""
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def format_phone(series):
    """Malaysian local format: digits only, 60->0, leading 6 dropped, leading 0 ensured.

    DECISION TO CONFIRM: the original had two versions of this -- the main path
    prepended a leading '0', the one inside email_phone() did not, so the CSV and
    the parquet disagreed. This keeps the prepend-'0' behaviour. Check against a
    sample of real numbers before the first production run. [C3]
    """
    s = series.apply(lambda x: x.decode('utf-8', errors='ignore') if isinstance(x, bytes) else x)
    s = s.astype(str).str.replace(r'\D+', '', regex=True)
    s = s.replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    s = s.str.replace(r'^60', '0', regex=True)
    s = s.str.replace(r'^6', '', regex=True)
    return s.apply(lambda x: x if pd.isna(x) or str(x).startswith('0') else '0' + str(x))


def decode_bytes(series):
    return series.apply(lambda x: x.decode('utf-8', errors='ignore') if isinstance(x, bytes) else x)


def merge_1to1(left, right, on='CIFNO', how='left', label=''):
    """Left-join that cannot change the row count. De-duplicates the right frame
    and drops its null keys first, then validates many-to-one."""
    before = len(left)
    right = right.dropna(subset=[on]).drop_duplicates(subset=on)
    out = left.merge(right, on=on, how=how, validate='many_to_one')
    if len(out) != before:
        raise ValueError(f"merge '{label}' changed row count: {before:,} -> {len(out):,}")
    return out

## 5. LoadersOne function per source. Each returns a frame keyed on `CIFNO` and renames by name.Nothing here writes to a global.

In [ ]:
def load_deposits():
    """CASA + Non-CASA, one row per CIFNO (latest account record wins).

    Returns (deposits_df, salary_tagged_cifs).
    ORIGINAL: read the CASA file a SECOND time later on just to get MRSFG4.
              Read once here instead.
    """
    casa = read_table(
        CONFIG['casa'],
        columns=['CIFNO', 'CFNA1', 'CFSSNO', 'AGE', 'CFINSC', 'Customer Type', 'CFSIC2',
                 'CFEMPL', 'CFSFLG', 'CBAL', 'CFBUST', 'BRANCH', 'DATOP7', 'MRSFG4'],
        rename={'CFNA1': 'NAME', 'CFSSNO': 'NRIC', 'CBAL': 'BAL',
                'BRANCH': 'br_id', 'DATOP7': 'DATOP'},
    )
    casa['CIFNO'] = clean_cif(casa['CIFNO'])

    # Salary tagging comes from the same read.
    salary_cifs = set(casa.loc[casa['MRSFG4'] == 'Y', 'CIFNO'].dropna())
    casa = casa.drop(columns='MRSFG4')

    nonc = read_table(
        CONFIG['nonc'],
        columns=['CIFNO', 'ACNAME', 'CFSSNO', 'AGE', 'CFINSC', 'Customer Type', 'CFSIC2',
                 'MthEnd Bal', 'CFBUST', 'BRN', 'ISSDT'],
        rename={'ACNAME': 'NAME', 'CFSSNO': 'NRIC', 'MthEnd Bal': 'BAL',
                'BRN': 'br_id', 'ISSDT': 'DATOP'},
    )
    nonc['CIFNO'] = clean_cif(nonc['CIFNO'])

    # Parse DATOP per source BEFORE concatenating.
    # ORIGINAL: sorted DATOP as a string across two sources whose date formats
    #           may differ, so "most recent record" was unreliable. [M4]
    for frame in (casa, nonc):
        frame['DATOP'] = pd.to_datetime(frame['DATOP'], errors='coerce', format='mixed')

    df = pd.concat([casa, nonc], ignore_index=True)

    unparsed = df['DATOP'].isna().sum()
    if unparsed:
        print(f"  note: {unparsed:,} of {len(df):,} DATOP values could not be parsed as dates")

    df = (df.sort_values(['CIFNO', 'DATOP'], ascending=[True, False], na_position='last')
            .drop_duplicates('CIFNO'))
    return df, salary_cifs


def eligible_cif_set(depo):
    """Individual + PDPA consent + adult + not staff, minus financing-file staff."""
    is_individual = (depo['Customer Type'] == 'Individual') & \
                    (depo['CFINSC'].isna() | (depo['CFINSC'] == ''))
    is_pdpa  = depo['CFSIC2'] == 'Y'
    is_adult = depo['AGE'].between(18, 100)

    # astype('string') so an all-NaN (float) column cannot blow up .str. [M3]
    employer_txt = depo['CFEMPL'].astype('string').fillna('')
    is_staff = (depo['CFSFLG'] == 'Y') | \
               employer_txt.str.contains('bank islam|BIMB|bankislam', case=False, regex=True, na=False)

    fin = read_table(CONFIG['fina'], columns=['CifNo', 'StaffFlag'], rename={'CifNo': 'CIFNO'})
    fin_staff = set(clean_cif(fin.loc[fin['StaffFlag'] == 'Y', 'CIFNO']).dropna())

    eligible = set(depo.loc[is_individual & is_pdpa & is_adult & ~is_staff, 'CIFNO'].dropna())
    return eligible - fin_staff

In [ ]:
def load_gold():
    df = read_table(CONFIG['gold'], columns=['CIF Number'], rename={'CIF Number': 'CIFNO'})
    df['CIFNO'] = clean_cif(df['CIFNO'])
    df = df.drop_duplicates('CIFNO')
    df['has_gold'] = 'Y'
    return df


def load_credit_card_flags():
    """has_cc plus the single best card type per customer."""
    cc = read_table(CONFIG['cc'])
    absent = sorted(set(['CIFNO', 'has_cc'] + CC_TYPE_ORDER) - set(cc.columns))
    if absent:
        raise KeyError(f"cc file missing expected column(s): {absent}. "
                       f"Update CC_TYPE_ORDER in §2 if a new card type was added.")

    cc = cc[['CIFNO', 'has_cc'] + CC_TYPE_ORDER]
    melted = cc.melt(id_vars=['CIFNO', 'has_cc'], var_name='cc_type', value_name='flag')
    melted = melted[melted['flag'] == 'Y'].copy()
    melted['cc_type'] = pd.Categorical(melted['cc_type'], categories=CC_TYPE_ORDER, ordered=True)

    out = (melted.sort_values(['CIFNO', 'cc_type'])
                 .drop_duplicates('CIFNO')[['CIFNO', 'has_cc', 'cc_type']])
    out['CIFNO'] = clean_cif(out['CIFNO'])
    return out


def load_employment():
    df = read_table(
        CONFIG['empl'],
        columns=['CIFNO', 'empl_name', 'package_tag(DealerCode)', 'empl_type_name'],
        rename={'empl_name': 'employer', 'package_tag(DealerCode)': 'package_employer',
                'empl_type_name': 'employer_type'},
    )
    df['CIFNO'] = clean_cif(df['CIFNO'])
    df = df.drop_duplicates('CIFNO')
    df['employer'] = (decode_bytes(df['employer']).astype(str)
                      .str.replace(r'[^a-zA-Z0-9\s]', '', regex=True))
    return df


def load_ibmb():
    df = read_table(
        CONFIG['ibmb'],
        columns=['CIFNO', 'has_ib', 'has_mb', 'Email Address', 'Phone Number'],
        rename={'Email Address': 'email_ib', 'Phone Number': 'phone_ib'},
    )
    df['CIFNO'] = clean_cif(df['CIFNO'])
    return df.drop_duplicates('CIFNO')


def load_unit_trust():
    df = read_table(
        CONFIG['ut_aum'],
        columns=['CIF Number', 'Total Holdings', 'AUM'],
        rename={'CIF Number': 'CIFNO', 'Total Holdings': 'unit_trust_balance', 'AUM': 'unit_trust_unit'},
    )
    df['CIFNO'] = clean_cif(df['CIFNO'])
    return df.groupby('CIFNO', as_index=False).sum()


def load_salary_crediting():
    df = read_table(CONFIG['salary_cr'], rename={'cust_id': 'CIFNO'})
    df['CIFNO'] = clean_cif(df['CIFNO'])
    df = df[['CIFNO']].drop_duplicates('CIFNO')
    df['salary_crediting_6mths'] = 'Y'
    return df


def load_debit_card():
    df = read_table(CONFIG['debit'], columns=['CIFNO', 'debitcard_type'])
    df['CIFNO'] = clean_cif(df['CIFNO'])
    return df.drop_duplicates('CIFNO')


def load_xsell():
    df = read_table(
        CONFIG['query_dwnld'],
        columns=['cust_id', 'current_products', 'product_recommendation', 'xsell_probability'],
        rename={'cust_id': 'CIFNO'},
    )
    df['CIFNO'] = clean_cif(df['CIFNO'])
    return df.drop_duplicates('CIFNO')


def load_product_holding():
    df = read_table(CONFIG['ph'], columns=['Cif', 'PH'],
                    rename={'Cif': 'CIFNO', 'PH': 'prod_holding'})
    df['CIFNO'] = clean_cif(df['CIFNO'])
    return df.drop_duplicates('CIFNO')


def load_cc_limit():
    df = read_table(
        CONFIG['cc_master'],
        columns=['Cif Number', 'Finance Type', 'Financing Amount'],
        rename={'Cif Number': 'CIFNO', 'Financing Amount': 'cc_credit_limit'},
    )
    df = df[df['Finance Type'] == 'CC'][['CIFNO', 'cc_credit_limit']]
    df['CIFNO'] = clean_cif(df['CIFNO'])
    return df.drop_duplicates('CIFNO')

In [ ]:
def load_contact_details():
    """Best phone + email per CIFNO from IBMB, financing and CASA.

    Uses the LAGGED contact_* paths from §2, not the current-month core files.
    Preference order: has a phone > phone starts 019 > has an email.
    """
    ibmb = read_table(
        CONFIG['ibmb'],
        columns=['CIFNO', 'Email Address', 'Phone Number'],
        rename={'Email Address': 'email', 'Phone Number': 'phone'},
    )
    ibmb['phone'] = format_phone(ibmb['phone'])

    def _melt_phones(path, rename):
        df = read_table(path, columns=list(rename), rename=rename)
        long = (df.melt(id_vars=['CIFNO', 'email'],
                        value_vars=['phone1', 'phone2', 'phone3'],
                        value_name='phone')
                  .drop(columns='variable'))
        long['phone'] = format_phone(long['phone'])
        return long

    fina_long = _melt_phones(CONFIG['contact_fina'], {
        'CifNo': 'CIFNO', 'PhoneNo1': 'phone1', 'PhoneNo2': 'phone2',
        'PhoneNo3': 'phone3', 'EmailAdd': 'email',
    })
    casa_long = _melt_phones(CONFIG['contact_casa'], {
        'CIFNO': 'CIFNO', 'CFHPHO': 'phone1', 'CFBPHO': 'phone2',
        'CFBFHO': 'phone3', 'CFCEML': 'email',
    })

    df = pd.concat([ibmb, fina_long, casa_long], ignore_index=True)
    df['CIFNO'] = clean_cif(df['CIFNO'])

    df['email'] = decode_bytes(df['email']).astype(str).str.strip().str.lower()
    df.loc[~df['email'].str.contains('@', na=False), 'email'] = np.nan
    df['email'] = df['email'].replace({'nan': np.nan, 'none': np.nan})

    df['_has_phone'] = df['phone'].notna().astype(int)
    df['_is_019']    = df['phone'].str.startswith('019', na=False).astype(int)
    df['_has_email'] = df['email'].notna().astype(int)

    df = (df.sort_values(['CIFNO', '_has_phone', '_is_019', '_has_email'],
                         ascending=[True, False, False, False])
            .drop_duplicates('CIFNO'))
    return df[['CIFNO', 'email', 'phone']].reset_index(drop=True)


FINANCE_COL_MAP = {
    'AR-RAHNU Value': 'AR-RAHNU Balance',
    'HFA Value': 'HFA Balance',
    'Personal Financing Value': 'Personal Financing Balance',
    'Vehicle Value': 'VF Balance',
    'VF Value': 'VF Balance',
    'CASATIA Value': 'CASATIA Balance',
    'Non-CASATIA Value': 'Non-CASATIA Balance',
}
FINANCE_COLS = ['AR-RAHNU Balance', 'HFA Balance', 'Personal Financing Balance',
                'VF Balance', 'CC Limit', 'CASATIA Balance', 'Non-CASATIA Balance']


def load_financial_summary(cif_frame):
    """Balances per CIFNO for the parquet deliverable.

    ORIGINAL: every read was wrapped in a bare `except Exception` that substituted
    an empty DataFrame, so a renamed column produced a file of zero balances with
    no error at all. Reads now fail loudly. [C4]
    ORIGINAL: aggfunc=np.sum -- removed in current pandas. [M1]
    """
    # --- financing facilities -> one column per FacilityType
    fina = read_table(CONFIG['fina'],
                      columns=['CifNo', 'FacilityType', 'OutstandingBalance'],
                      rename={'CifNo': 'CIFNO', 'OutstandingBalance': 'Value'})
    fina['CIFNO'] = clean_cif(fina['CIFNO'])
    fina_pvt = pd.pivot_table(fina, values='Value', index='CIFNO',
                              columns='FacilityType', aggfunc='sum')
    # names derived from the data itself, not a blind positional list
    fina_pvt.columns = [f'{c} Value' for c in fina_pvt.columns]
    fina_pvt = fina_pvt.fillna(0).reset_index()

    # --- deposits
    casa = read_table(CONFIG['casa'], columns=['CIFNO', 'CBAL'], rename={'CBAL': 'Value'})
    casa['ProdType'] = 'CASATIA'
    nonc = read_table(CONFIG['nonc'], columns=['CIFNO', 'MthEnd Bal'], rename={'MthEnd Bal': 'Value'})
    nonc['ProdType'] = 'Non-CASATIA'

    depo = pd.concat([casa, nonc], ignore_index=True)
    depo['CIFNO'] = clean_cif(depo['CIFNO'])
    depo_pvt = pd.pivot_table(depo, values='Value', index='CIFNO',
                              columns='ProdType', aggfunc='sum')
    depo_pvt.columns = [f'{c} Value' for c in depo_pvt.columns]
    depo_pvt = depo_pvt.fillna(0).reset_index()

    # --- credit card limit
    cc = read_table(CONFIG['cc_master'],
                    columns=['Cif Number', 'Financing Amount'],
                    rename={'Cif Number': 'CIFNO', 'Financing Amount': 'CC Limit'})
    cc['CIFNO'] = clean_cif(cc['CIFNO'])
    cc = cc.groupby('CIFNO', as_index=False)['CC Limit'].sum()

    base = cif_frame[['CIFNO']].drop_duplicates()
    out = (base.merge(depo_pvt, on='CIFNO', how='left')
                .merge(fina_pvt, on='CIFNO', how='left')
                .merge(cc,       on='CIFNO', how='left')
                .rename(columns=FINANCE_COL_MAP))

    absent = [c for c in FINANCE_COLS if c not in out.columns]
    if absent:
        # Kept as zero columns for schema stability, but SAY SO rather than
        # silently shipping zeros.
        print(f"  WARNING: no data for {absent} -- these columns will be all zero")
        for c in absent:
            out[c] = 0

    return out[FINANCE_COLS + ['CIFNO']].fillna(0).groupby('CIFNO', as_index=False).sum()

## 6. Base deposits and eligibility

In [ ]:
deposits, salary_cifs = load_deposits()
eligible_cifs = eligible_cif_set(deposits)

print(f"deposit records (unique CIFNO) : {len(deposits):,}")
print(f"sale-eligible                  : {len(eligible_cifs):,} "
      f"({len(eligible_cifs) / len(deposits):.1%})")
print(f"salary-tagged (MRSFG4 = Y)     : {len(salary_cifs):,}")

## 7. Branch and occupation lookupsCustomers whose `br_id` is absent from the branch lookup end up with no branch and aredropped in §9. The original did that silently; the count is printed there now. **[C5]**

In [ ]:
branch_lookup = read_table(CONFIG['branch_lookup'])
occ_lookup    = read_table(CONFIG['occ_lookup']).drop_duplicates('CFBUST')

# Latest-dated account decides the branch when a CIF has several.
branches = (deposits[['CIFNO', 'br_id', 'DATOP']]
            .merge(branch_lookup, on='br_id', how='inner')
            .sort_values(['CIFNO', 'DATOP'], ascending=[True, False], na_position='last')
            .drop_duplicates('CIFNO')[['CIFNO', 'branch', 'state', 'region']])

leads = deposits.merge(occ_lookup, on='CFBUST', how='left', validate='many_to_one')
leads = merge_1to1(leads, branches, label='branch')
leads['sale_eligibility'] = np.where(leads['CIFNO'].isin(eligible_cifs), 'Y', 'N')
leads['salary_tagging']   = np.where(leads['CIFNO'].isin(salary_cifs), 'Y', 'N')

print(f"rows: {len(leads):,}   with branch: {leads['branch'].notna().sum():,}")

## 8. Feature mergesEvery join goes through `merge_1to1`, so none of these can change the row count.

In [ ]:
feature_sources = [
    ('gold',            load_gold),
    ('credit_card',     load_credit_card_flags),
    ('employment',      load_employment),
    ('ibmb',            load_ibmb),
    ('salary_crediting',load_salary_crediting),
    ('debit_card',      load_debit_card),
    ('xsell',           load_xsell),
    ('product_holding', load_product_holding),
    ('cc_limit',        load_cc_limit),
    ('unit_trust',      load_unit_trust),
]

for label, loader in feature_sources:
    frame = loader()
    frame['CIFNO'] = clean_cif(frame['CIFNO'])
    leads = merge_1to1(leads, frame, label=label)
    print(f"  merged {label:<17s} rows={len(leads):,}")

## 9. Formatting, dtypes and PII maskingBehaviour changes from the original to verify on the first run:- **Age groups.** The original binned from `0` but labelled the first bucket `18-24`, so  under-18s were labelled adults, and the `AGE` floor was applied *after* binning so it never  reached the bin. Age is floored first and there is now an explicit `<18` bucket. **[C7]**- **`xsell_probability`** is cast to `float64`. It was `object` — a probability stored as  text, which consumers could not sort or threshold. **[M10]**- **`age_group` and `cc_type`** are plain strings in both deliverables. They were `category`  in the parquet and `object` in the CSV, and the null-fill step skipped categoricals. **[M13]**- **`CIFNO`** is `Int64` in both. It was `int64` in the CSV and `Int64` in the parquet. **[M14]**

In [ ]:
FINAL_COLS = [
    'month', 'channel', 'CIFNO', 'sale_eligibility', 'NAME', 'NRIC', 'AGE', 'age_group',
    'Occupation', 'employer', 'package_employer', 'employer_type', 'has_gold', 'has_cc',
    'cc_type', 'unit_trust_balance', 'unit_trust_unit', 'has_ib', 'has_mb', 'salary_tagging',
    'salary_crediting_6mths', 'debitcard_type', 'current_products', 'product_recommendation',
    'xsell_probability', 'prod_holding', 'cc_credit_limit', 'branch', 'state', 'region',
    'email', 'phone',
]
PII_COLS = ['name', 'nric', 'email', 'phone', 'employer']

# --- drop rows with no branch, and SAY how many                        [C5]
before = len(leads)
leads = leads.drop_duplicates('CIFNO').dropna(subset=['branch']).copy()
print(f"dropped {before - len(leads):,} rows with no branch-lookup match "
      f"({before:,} -> {len(leads):,})")

# --- Y/N flags
for col in ['has_gold', 'has_cc', 'has_ib', 'has_mb', 'salary_crediting_6mths']:
    if col in leads.columns:
        leads[col] = leads[col].fillna('N').astype(str).str.upper()

leads['package_employer'] = np.where(leads['package_employer'] == 'Package',
                                     'Package', 'Non Package')

# --- contact + derived fields
leads['phone']   = format_phone(leads['phone_ib'])
leads['email']   = decode_bytes(leads['email_ib']).fillna('')
leads['month']   = dt.now().strftime('%B %Y')
leads['channel'] = ''          # always blank -- populate or drop. [M15]

# --- age: floor FIRST, then bin                                        [C7]
leads['AGE'] = pd.to_numeric(leads['AGE'], errors='coerce').fillna(0).astype(int).clip(lower=1)
leads['age_group'] = pd.cut(
    leads['AGE'],
    bins=[0, 17, 24, 34, 44, 54, 64, 150],
    labels=['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65+'],
)

# --- consistent dtypes across both deliverables            [M10][M13][M14]
leads['CIFNO']             = clean_cif(leads['CIFNO'])
leads['xsell_probability'] = pd.to_numeric(leads['xsell_probability'], errors='coerce')
leads['age_group']         = leads['age_group'].astype('object')
leads['cc_type']           = leads['cc_type'].astype('object')

# --- final column set, lower-cased except CIFNO
leads = leads.reindex(columns=FINAL_COLS)
leads.columns = [c if c == 'CIFNO' else c.lower() for c in leads.columns]

# --- mask PII for ineligible customers
# NRIC and phone are often read as int64, which loses leading zeros AND makes
# the 'MASKED' assignment raise on pandas 3. Force them to text first.
for col in PII_COLS:
    leads[col] = leads[col].astype(object).map(lambda v: '' if pd.isna(v) else str(v))
leads.loc[leads['sale_eligibility'] == 'N', PII_COLS] = 'MASKED'

# --- null fill
obj_cols = leads.select_dtypes(include='object').columns
num_cols = leads.select_dtypes(include='number').columns
leads[obj_cols] = leads[obj_cols].fillna('')
leads[num_cols] = leads[num_cols].fillna(0)

print(f"\nfinal lead base: {leads.shape[0]:,} rows x {leads.shape[1]} columns")
print(leads['sale_eligibility'].value_counts().to_string())

## 10. Deliverable A — CSV (deposit-oriented)All customers, PII masked for the ineligible, plus 17 deposit columns from `depo_info`.The split now uses `np.array_split` on the frame actually being written. The original took`len()` of the *pre-merge* frame and sliced the post-merge one, so the halves were uneven —and could leave `p2` nearly empty. **[C2]** Filenames carry `run_date` instead of ahardcoded `20260910`. **[C6]**

In [ ]:
depo_info = read_table(CONFIG['depo_info'])
depo_info['CIFNO'] = clean_cif(depo_info['CIFNO'])
depo_info = depo_info.dropna(subset=['CIFNO']).drop_duplicates('CIFNO')

csv_df = leads.merge(depo_info, on='CIFNO', how='inner', validate='one_to_one')
print(f"depo_info matched {len(csv_df):,} of {len(leads):,} leads "
      f"({len(csv_df) / len(leads):.1%})")

# Even split of the frame actually being written -- works for any csv_parts.
bounds = np.linspace(0, len(csv_df), CONFIG['csv_parts'] + 1).round().astype(int)
for i in range(CONFIG['csv_parts']):
    part = csv_df.iloc[bounds[i]:bounds[i + 1]]
    out_path = os.path.join(
        CONFIG['output_dir'],
        f"leadgen_uploadfile_{CONFIG['run_date']}_p{i + 1}.csv")
    part.to_csv(out_path, index=False, header=True, encoding='utf-8')
    print(f"  wrote {len(part):,} rows -> {os.path.basename(out_path)}")

## 11. Deliverable B — Parquet (financing / cross-sell)Eligible customers only, virtual branches excluded, with re-derived contact details and7 financing balance columns.**This file carries unmasked name, NRIC, email and phone.** Handle and store it accordingly.

In [ ]:
pq_df = leads[(leads['sale_eligibility'] == 'Y') & (leads['branch'] != '')].copy()
pq_df = pq_df[~pq_df['branch'].isin(CONFIG['parquet_excluded_branches'])].reset_index(drop=True)
print(f"eligible, branch-filtered: {len(pq_df):,} rows")

# contact details from the lagged extracts replace the main-path ones
pq_df = pq_df.drop(columns=['email', 'phone'])
pq_df = merge_1to1(pq_df, load_contact_details(), label='contact')
pq_df = merge_1to1(pq_df, load_financial_summary(pq_df), label='financials')

if CONFIG['normalise_parquet_names']:
    pq_df = pq_df.rename(columns={c: c.lower().replace(' ', '_').replace('-', '_')
                                  for c in FINANCE_COLS})

out_path = os.path.join(CONFIG['output_dir'],
                        f"leadgen_uploadfile_{CONFIG['run_date']}.parquet")
pq_df.to_parquet(out_path, engine='fastparquet', compression='GZIP')
print(f"  wrote {len(pq_df):,} rows x {pq_df.shape[1]} cols -> {os.path.basename(out_path)}")

## 12. Optional guard rails**Delete this cell if you don't want it.** It is four assertions that cost nothing and coverthe failure modes this pipeline is prone to: a silently-empty upstream load producingall-zero balances, a column swap putting emails in the phone field, and PII shippingunmasked. Without them a broken input produces a plausible-looking file that nobodyquestions until the sales team calls the wrong people.

In [ ]:
# --- CSV deliverable
assert csv_df['CIFNO'].notna().all(),                      "CSV: null CIFNO"
assert not csv_df['CIFNO'].duplicated().any(),             "CSV: duplicate CIFNO"
assert (csv_df.loc[csv_df['sale_eligibility'] == 'N', 'nric'] == 'MASKED').all(), \
       "CSV: unmasked NRIC on an ineligible customer"

# --- parquet deliverable
assert pq_df['CIFNO'].notna().all(),                       "PQ: null CIFNO"
assert not pq_df['CIFNO'].duplicated().any(),              "PQ: duplicate CIFNO"
assert (pq_df['sale_eligibility'] == 'Y').all(),           "PQ: ineligible customer present"
assert pq_df[FINANCE_COLS].sum().gt(0).any(),              "PQ: every balance column is zero"

_mail = pq_df['email'].dropna()
_mail = _mail[_mail != '']
assert _mail.str.contains('@').all(),                      "PQ: non-email value in email column"

print("all checks passed")